In [ ]:
# HAM10000 — Ablation C6
import os, random
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models, datasets
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler

from transformers import CLIPTokenizer, CLIPTextModel
from torchinfo import summary
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    accuracy_score, roc_curve, auc
)
from scipy import stats as scipy_stats
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
os.environ["PYTHONHASHSEED"]       = str(SEED)

In [ ]:
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16
MAX_LEN    = 32

TRAIN_DIR = "/kaggle/input/datasets/sujansarkar/ham10000-folder-wise/ham10000_80_20/ham10000_80_20/train_dir"
TEST_DIR  = "/kaggle/input/datasets/sujansarkar/ham10000-folder-wise/ham10000_80_20/ham10000_80_20/test_dir"
META_CSV  = "/kaggle/input/datasets/sujansarkar/ham10000-folder-wise/HAM10000_metadata.csv"

train_tfms = transforms.Compose([
    transforms.Resize(288),
    transforms.RandomResizedCrop(256, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomApply(
        [transforms.ColorJitter(brightness=0.2, contrast=0.2, hue=0.05)],
        p=0.5
    ),
    transforms.RandomAffine(
        degrees=180, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=15
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_tfms = transforms.Compose([
    transforms.Resize(288),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print(f"Device: {DEVICE}")

In [ ]:
full_train_folder = datasets.ImageFolder(TRAIN_DIR)
test_folder       = datasets.ImageFolder(TEST_DIR)

all_paths  = [p for p, _ in full_train_folder.imgs]
all_labels = [l for _, l in full_train_folder.imgs]

class_to_idx = full_train_folder.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
NUM_CLASSES  = len(class_to_idx)

# Same 80/20 split across all C2–C6 for consistency
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(sss.split(all_paths, all_labels))

train_paths  = [all_paths[i]  for i in train_idx]
train_labels = [all_labels[i] for i in train_idx]
val_paths    = [all_paths[i]  for i in val_idx]
val_labels   = [all_labels[i] for i in val_idx]

test_paths  = [p for p, _ in test_folder.imgs]
test_labels = [l for _, l in test_folder.imgs]

print(f"Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")
print(f"Classes ({NUM_CLASSES}): {class_to_idx}")

In [ ]:
NUMERICAL_COLS   = ['age']
CATEGORICAL_COLS = ['sex', 'localization']

meta_df = pd.read_csv(META_CSV).set_index('image_id')

train_ids  = [os.path.splitext(os.path.basename(p))[0] for p in train_paths]
train_meta = meta_df.loc[meta_df.index.intersection(train_ids)]
age_mean   = train_meta['age'].mean()
age_std    = train_meta['age'].std() + 1e-6

meta_df['age'] = meta_df['age'].fillna(age_mean)
meta_df['age'] = (meta_df['age'] - age_mean) / age_std

meta_df['sex']          = meta_df['sex'].fillna('unknown')
meta_df['localization'] = meta_df['localization'].fillna('unknown')

train_meta_norm = meta_df.loc[meta_df.index.intersection(train_ids)]
cat_dummies     = pd.get_dummies(
    train_meta_norm[CATEGORICAL_COLS], prefix=CATEGORICAL_COLS
)
CAT_COLUMNS = cat_dummies.columns.tolist()
NUM_META    = len(NUMERICAL_COLS) + len(CAT_COLUMNS)

def get_meta_vector(image_id):
    if image_id not in meta_df.index:
        return torch.zeros(NUM_META, dtype=torch.float32)
    row = meta_df.loc[image_id]
    num = [float(row['age'])]
    cat = pd.get_dummies(
        pd.DataFrame([row[CATEGORICAL_COLS].to_dict()]),
        prefix=CATEGORICAL_COLS
    ).reindex(columns=CAT_COLUMNS, fill_value=0).values[0].tolist()
    return torch.tensor(num + cat, dtype=torch.float32)

print(f"Metadata feature dimensionality: {NUM_META}")
print(f"Categorical columns: {CAT_COLUMNS}")

In [ ]:
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
DEFAULT_PROMPT = "A dermoscopic image of a skin lesion."

def generate_prompt(image_id):
    if image_id not in meta_df.index:
        return DEFAULT_PROMPT
    row          = meta_df.loc[image_id]
    age_norm     = row.get('age', None)
    sex          = str(row.get('sex', 'unknown')).lower()
    localization = str(row.get('localization', 'unknown')).lower()
    age_years    = int(round(float(age_norm) * age_std + age_mean)) \
                   if age_norm is not None else None
    parts = ["A dermoscopic image of a skin lesion"]
    if localization != 'unknown':
        parts[0] += f" on the {localization}"
    if age_years and sex != 'unknown':
        parts.append(f"of a {age_years}-year-old {sex}")
    elif age_years:
        parts.append(f"of a {age_years}-year-old patient")
    elif sex != 'unknown':
        parts.append(f"of a {sex} patient")
    return " ".join(parts) + "."

def tokenise(prompt):
    enc = tokenizer(
        prompt, padding="max_length", max_length=MAX_LEN,
        truncation=True, return_tensors="pt"
    )
    return enc["input_ids"].squeeze(0), enc["attention_mask"].squeeze(0)

sample_prompt = generate_prompt(train_ids[0])
ids, mask = tokenise(sample_prompt)
print(f"Sample prompt : {sample_prompt}")
print(f"Token shape   : {ids.shape}")

In [ ]:
class HAMTriModalDataset(Dataset):
    def __init__(self, img_paths, labels, transform=None):
        self.img_paths = img_paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        path     = self.img_paths[idx]
        image_id = os.path.splitext(os.path.basename(path))[0]
    
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
    
        tab   = get_meta_vector(image_id)
    
        prompt              = generate_prompt(image_id)
        input_ids, attn_mask = tokenise(prompt)
    
        label = self.labels[idx]
        return image, input_ids, attn_mask, tab, label


train_dataset = HAMTriModalDataset(train_paths, train_labels, train_tfms)
val_dataset   = HAMTriModalDataset(val_paths,   val_labels,   val_test_tfms)
test_dataset  = HAMTriModalDataset(test_paths,  test_labels,  val_test_tfms)

In [ ]:
class_counts         = Counter(train_labels)
total_samples        = sum(class_counts.values())
raw_weights          = [
    total_samples / (NUM_CLASSES * class_counts[i] ** 0.5)
    for i in range(NUM_CLASSES)
]
mean_w               = sum(raw_weights) / NUM_CLASSES
class_weights_tensor = torch.FloatTensor([w / mean_w for w in raw_weights])


def seed_worker(worker_id):
    seed = SEED + worker_id
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True,
    persistent_workers=True, worker_init_fn=seed_worker,
    generator=torch.Generator().manual_seed(SEED)
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True,
    worker_init_fn=seed_worker
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True,
    worker_init_fn=seed_worker
)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
IDX_TO_CLASS = {
    class_to_idx["akiec"]: "Actinic Keratoses and Intraepithelial Carcinoma (AKIEC)",
    class_to_idx["bcc"]  : "Basal Cell Carcinoma (BCC)",
    class_to_idx["bkl"]  : "Benign Keratosis-like Lesions (BKL)",
    class_to_idx["df"]   : "Dermatofibroma (DF)",
    class_to_idx["mel"]  : "Melanoma (MEL)",
    class_to_idx["nv"]   : "Melanocytic Nevi (NV)",
    class_to_idx["vasc"] : "Vascular Lesions (VASC)",
}
class_names = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]

print(f"Classes ({NUM_CLASSES}):", class_names)
print("\nFocalLoss weights:", {
    IDX_TO_CLASS[i]: f"{class_weights_tensor[i]:.3f}"
    for i in range(NUM_CLASSES)
})
print("\nTrain class counts:", {
    IDX_TO_CLASS[k]: v for k, v in class_counts.items()
})

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, num_meta_features, num_classes=6, pretrained=True, freeze_backbone=False):
        super().__init__()

        # Vision Branch A: ConvNeXt-Tiny
        convnext = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None)
        self.convnext_features = convnext.features
        self.convnext_dim = 768

        # Vision Branch B: Swin-Tiny
        swin = models.swin_t(weights=models.Swin_T_Weights.DEFAULT if pretrained else None)
        self.swin_features = swin.features
        self.swin_norm = swin.norm
        self.swin_dim = swin.head.in_features  # 768

        # Vision Fusion Bottleneck — 768+768=1536 → 1024
        self.vision_fusion_dim = 1024
        self.vision_fuse = nn.Sequential(
            nn.Linear(self.convnext_dim + self.swin_dim, self.vision_fusion_dim),
            nn.LayerNorm(self.vision_fusion_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

        # Text Branch: CLIP
        self.text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")
        self.text_dim = self.text_encoder.config.hidden_size  # 512

        # Shared projection space
        self.embed_dim = 256
        self.vis_proj  = nn.Linear(self.vision_fusion_dim, self.embed_dim)
        self.text_proj = nn.Linear(self.text_dim, self.embed_dim)

        # Learnable positional encoding for the 8x8=64 vision patch grid
        self.pos_embed = nn.Parameter(torch.randn(1, 64, self.embed_dim) * 0.02)

        # Bidirectional cross-attention
        self.text_to_vis_attn = nn.MultiheadAttention(
            embed_dim=self.embed_dim, num_heads=8, dropout=0.1, batch_first=True
        )
        self.vis_to_text_attn = nn.MultiheadAttention(
            embed_dim=self.embed_dim, num_heads=8, dropout=0.1, batch_first=True
        )

        # Clinical Branch: Tabular MLP
        self.tabular_mlp = nn.Sequential(
            nn.Linear(num_meta_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )
        self.tabular_dim = 128
        self.meta_proj = nn.Linear(self.tabular_dim, self.embed_dim)

        # Joint multimodal gate — sees all 5 streams concatenated so it can
        # make relative comparisons between modalities per sample
        self.num_modalities = 5
        self.modal_gate = nn.Linear(self.embed_dim * self.num_modalities, self.num_modalities)

        # Final Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(self.embed_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

        if freeze_backbone:
            for p in self.convnext_features.parameters(): p.requires_grad = False
            for p in self.swin_features.parameters():     p.requires_grad = False
            for p in self.swin_norm.parameters():         p.requires_grad = False
            for p in self.text_encoder.parameters():      p.requires_grad = False

    def forward(self, x, input_ids, attention_mask, meta_data, return_attn=False):
        # Vision
        f_convnext = self.convnext_features(x).flatten(2).transpose(1, 2)  # [B, 64, 768]
        f_swin     = self.swin_norm(self.swin_features(x)).flatten(1, 2)   # [B, 64, 768]
        fused_vision = self.vision_fuse(torch.cat([f_convnext, f_swin], dim=-1))  # [B, 64, 1024]

        # Text
        text_feats = self.text_encoder(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state  # [B, 77, 512]

        # Project to shared space
        v_emb = F.normalize(self.vis_proj(fused_vision) + self.pos_embed, p=2, dim=-1)  # [B, 64, 256]
        t_emb = F.normalize(self.text_proj(text_feats),                   p=2, dim=-1)  # [B, 77, 256]

        # Bidirectional cross-attention — names reflect which stream is the query
        # text_to_vis: query=text → output has text sequence length [B, 77, 256]
        t_attended, attn_weights = self.text_to_vis_attn(query=t_emb, key=v_emb, value=v_emb)
        # vis_to_text: query=vision → output has vision sequence length [B, 64, 256]
        v_attended, _ = self.vis_to_text_attn(query=v_emb, key=t_emb, value=t_emb)

        t_attended_pooled = t_attended.mean(dim=1)  # [B, 256] — text enriched with visual context
        v_attended_pooled = v_attended.mean(dim=1)  # [B, 256] — vision enriched with text context
        global_text       = t_emb.mean(dim=1)       # [B, 256]
        global_vision     = v_emb.mean(dim=1)       # [B, 256]

        # Tabular
        f_meta      = self.tabular_mlp(meta_data)
        f_meta_proj = self.meta_proj(f_meta)         # [B, 256]

        # Joint gate — concatenate all streams so gate can compare them against each other
        modalities   = [t_attended_pooled, v_attended_pooled, global_text, global_vision, f_meta_proj]
        stacked      = torch.stack(modalities, dim=1)                          # [B, 5, 256]
        gate_weights = self.modal_gate(
            torch.cat(modalities, dim=-1)                                      # [B, 1280]
        ).softmax(dim=-1).unsqueeze(-1)                                        # [B, 5, 1]
        fused        = (stacked * gate_weights).sum(dim=1)                     # [B, 256]

        preds = self.classifier(fused)

        if return_attn:
            return preds, attn_weights
        return preds


model = FusionNet(
    num_meta_features=NUM_META,
    num_classes=NUM_CLASSES,
    pretrained=True,
    freeze_backbone=True
).to(DEVICE)

summary(
    model,
    input_size=[
        (BATCH_SIZE, 3, 256, 256),
        (BATCH_SIZE, 77),
        (BATCH_SIZE, 77),
        (BATCH_SIZE, NUM_META)
    ],
    dtypes=[torch.float32, torch.long, torch.long, torch.float32],
    col_names=["input_size", "output_size", "num_params"],
    depth=3
)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.ce    = nn.CrossEntropyLoss(
            weight=weight,
            label_smoothing=label_smoothing,
            reduction="none"
        )

    def forward(self, inputs, targets):
        ce_loss    = self.ce(inputs, targets)
        pt         = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


criterion = FocalLoss(
    weight=class_weights_tensor.to(DEVICE),
    gamma=2.0,
    label_smoothing=0.1
)
scaler = GradScaler("cuda")

In [ ]:
def compute_specificity(y_true, y_pred, num_classes):
    cm    = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    specs = []
    for i in range(num_classes):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)
        specs.append(TN / (TN + FP) if (TN + FP) > 0 else 0.0)
    return float(np.mean(specs))


def mixup_batch(images, tab, labels, alpha=0.3):
    lam      = np.random.beta(alpha, alpha)
    rand_idx = torch.randperm(images.size(0), device=images.device)
    mix_img  = lam * images + (1 - lam) * images[rand_idx]
    mix_tab  = lam * tab    + (1 - lam) * tab[rand_idx]
    return mix_img, mix_tab, labels, labels[rand_idx], lam


def train_one_epoch(model, loader, optimizer, criterion, device,
                    use_mixup=True):
    model.train()
    running_loss = 0.0
    y_true, y_pred = [], []

    for batch in tqdm(loader, desc="Training", leave=False):
        images, input_ids, attn_mask, tab, labels = [
            b.to(device) for b in batch
        ]

        if use_mixup:
            mix_img, mix_tab, y_a, y_b, lam = mixup_batch(
                images, tab, labels
            )
        else:
            mix_img, mix_tab, y_a, y_b, lam = images, tab, labels, labels, 1.0

        optimizer.zero_grad()
        with autocast("cuda"):
            out  = model(mix_img, input_ids, attn_mask, mix_tab)
            loss = (lam * criterion(out, y_a) +
                    (1 - lam) * criterion(out, y_b))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        y_true.extend((y_a if lam >= 0.5 else y_b).cpu().numpy())
        y_pred.extend(torch.argmax(out, 1).cpu().numpy())

    n = len(y_true)
    return (
        running_loss / n,
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, average="macro", zero_division=0),
        recall_score(y_true, y_pred,    average="macro", zero_division=0),
        f1_score(y_true, y_pred,        average="macro", zero_division=0),
        compute_specificity(y_true, y_pred, NUM_CLASSES)
    )


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval", leave=False):
            images, input_ids, attn_mask, tab, labels = [
                b.to(device) for b in batch
            ]
            out  = model(images, input_ids, attn_mask, tab)
            loss = criterion(out, labels)
            running_loss += loss.item() * images.size(0)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(torch.argmax(out, 1).cpu().numpy())

    n = len(y_true)
    return (
        running_loss / n,
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, average="macro", zero_division=0),
        recall_score(y_true, y_pred,    average="macro", zero_division=0),
        f1_score(y_true, y_pred,        average="macro", zero_division=0),
        compute_specificity(y_true, y_pred, NUM_CLASSES),
        y_true, y_pred
    )

In [ ]:
PHASE1_EPOCHS = 20

optimizer_p1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-2
)
scheduler_p1 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_p1, T_0=10, T_mult=2, eta_min=1e-6
)

train_losses, val_losses = [], []
best_val_f1, best_model_state = 0.0, None

print("=" * 60)
print("PHASE 1 — Frozen backbone | HAM10000 Model 2 C6")
print("=" * 60)

for epoch in range(PHASE1_EPOCHS):
    print(f"\nPhase 1 Epoch [{epoch+1}/{PHASE1_EPOCHS}]")
    tr = train_one_epoch(model, train_loader, optimizer_p1, criterion, DEVICE)
    vl = evaluate(model, val_loader, criterion, DEVICE)
    train_losses.append(tr[0]); val_losses.append(vl[0])
    scheduler_p1.step()

    print(f"LR: {optimizer_p1.param_groups[0]['lr']:.2e}")
    print(f"Train | Loss:{tr[0]:.4f} Acc:{tr[1]:.4f} "
          f"Prec:{tr[2]:.4f} Rec:{tr[3]:.4f} F1:{tr[4]:.4f} Spec:{tr[5]:.4f}")
    print(f"Val   | Loss:{vl[0]:.4f} Acc:{vl[1]:.4f} "
          f"Prec:{vl[2]:.4f} Rec:{vl[3]:.4f} F1:{vl[4]:.4f} Spec:{vl[5]:.4f}")

    if vl[4] > best_val_f1:
        best_val_f1      = vl[4]
        best_model_state = model.state_dict()
        print(f"  Val F1 improved ({best_val_f1:.4f}) — checkpoint saved")

model.load_state_dict(best_model_state)
print(f"\nPhase 1 complete. Best val F1: {best_val_f1:.4f}")

In [ ]:
for p in model.parameters():
    p.requires_grad = True

PHASE2_EPOCHS = 60
BACKBONE_LR   = 1e-6
HEAD_LR       = 1e-4

optimizer_p2 = torch.optim.AdamW([
    {"params": model.convnext_features.parameters(), "lr": BACKBONE_LR},
    {"params": model.swin_features.parameters(),     "lr": BACKBONE_LR},
    {"params": model.swin_norm.parameters(),         "lr": BACKBONE_LR},
    {"params": model.text_encoder.parameters(),      "lr": BACKBONE_LR},
    {"params": model.vision_fuse.parameters(),       "lr": HEAD_LR},
    {"params": model.vis_proj.parameters(),          "lr": HEAD_LR},
    {"params": [model.pos_embed],                    "lr": HEAD_LR},
    {"params": model.text_proj.parameters(),         "lr": HEAD_LR},
    {"params": model.text_to_vis_attn.parameters(),  "lr": HEAD_LR},
    {"params": model.vis_to_text_attn.parameters(),  "lr": HEAD_LR},
    {"params": model.tabular_mlp.parameters(),       "lr": HEAD_LR},
    {"params": model.meta_proj.parameters(),         "lr": HEAD_LR},
    {"params": model.modal_gate.parameters(),        "lr": HEAD_LR},
    {"params": model.classifier.parameters(),        "lr": HEAD_LR},
], weight_decay=5e-2)

scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_p2, T_0=10, T_mult=2, eta_min=1e-7
)

fine_train_losses, fine_val_losses = [], []
best_val_f1, best_model_state = 0.0, None
patience, counter = 15, 0

print("=" * 60)
print("PHASE 2 — Full fine-tuning | HAM10000 Model 2 C6")
print(f"Backbone LR: {BACKBONE_LR:.0e} | Head LR: {HEAD_LR:.0e}")
print("=" * 60)

for epoch in range(PHASE2_EPOCHS):
    print(f"\nPhase 2 Epoch [{epoch+1}/{PHASE2_EPOCHS}]")
    tr = train_one_epoch(model, train_loader, optimizer_p2, criterion, DEVICE)
    vl = evaluate(model, val_loader, criterion, DEVICE)
    fine_train_losses.append(tr[0]); fine_val_losses.append(vl[0])
    scheduler_p2.step()

    print(f"LR (backbone):{optimizer_p2.param_groups[0]['lr']:.2e} | "
          f"LR (head):{optimizer_p2.param_groups[4]['lr']:.2e}")
    print(f"Train | Loss:{tr[0]:.4f} Acc:{tr[1]:.4f} "
          f"Prec:{tr[2]:.4f} Rec:{tr[3]:.4f} F1:{tr[4]:.4f} Spec:{tr[5]:.4f}")
    print(f"Val   | Loss:{vl[0]:.4f} Acc:{vl[1]:.4f} "
          f"Prec:{vl[2]:.4f} Rec:{vl[3]:.4f} F1:{vl[4]:.4f} Spec:{vl[5]:.4f}")

    if vl[4] > best_val_f1:
        best_val_f1      = vl[4]
        best_model_state = model.state_dict()
        torch.save(best_model_state,
                   "/kaggle/working/HAM10000_Model2_C6_FullFusionNet.pth")
        counter = 0
        print(f"  Val F1 improved ({best_val_f1:.4f}) — saved")
    else:
        counter += 1
        print(f"  No improvement. Counter: {counter}/{patience}")

    if counter >= patience:
        print("\nEarly stopping triggered.")
        break

model.load_state_dict(best_model_state)
print(f"\nPhase 2 complete. Best val F1: {best_val_f1:.4f}")

In [ ]:
unweighted_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
    worker_init_fn=seed_worker
)

for pg in optimizer_p2.param_groups:
    pg['lr'] = 1e-6

PHASE3_EPOCHS = 10
calib_train_losses, calib_val_losses = [], []

print("=" * 60)
print("PHASE 3 — Calibration (LR locked 1e-6, no MixUp)")
print("=" * 60)

for epoch in range(PHASE3_EPOCHS):
    print(f"\nPhase 3 Epoch [{epoch+1}/{PHASE3_EPOCHS}]")
    tr = train_one_epoch(
        model, unweighted_loader, optimizer_p2,
        criterion, DEVICE, use_mixup=False
    )
    vl = evaluate(model, val_loader, criterion, DEVICE)
    calib_train_losses.append(tr[0]); calib_val_losses.append(vl[0])

    print(f"Train | Loss:{tr[0]:.4f} Acc:{tr[1]:.4f} F1:{tr[4]:.4f}")
    print(f"Val   | Loss:{vl[0]:.4f} Acc:{vl[1]:.4f} F1:{vl[4]:.4f}")

    if vl[4] > best_val_f1:
        best_val_f1      = vl[4]
        best_model_state = model.state_dict()
        print(f"  Phase 3 val F1 improved ({best_val_f1:.4f}) — saved")

if best_model_state:
    model.load_state_dict(best_model_state)
print(f"\nBest overall val F1: {best_val_f1:.4f}")

In [ ]:
tr_loss, tr_acc, _, _, _, tr_spec, tr_true, tr_pred = evaluate(
    model, train_loader, criterion, DEVICE
)
print("\nTRAIN REPORT")
print(f"Loss:{tr_loss:.4f} | Acc:{tr_acc:.4f} | Spec:{tr_spec:.4f}")
print(classification_report(
    tr_true, tr_pred, target_names=class_names, zero_division=0, digits=4
))
print(f"Macro F1:    {f1_score(tr_true, tr_pred, average='macro',    zero_division=0):.4f}")
print(f"Weighted F1: {f1_score(tr_true, tr_pred, average='weighted', zero_division=0):.4f}")

vl_loss, vl_acc, _, _, _, vl_spec, vl_true, vl_pred = evaluate(
    model, val_loader, criterion, DEVICE
)
print("\nVALIDATION REPORT")
print(f"Loss:{vl_loss:.4f} | Acc:{vl_acc:.4f} | Spec:{vl_spec:.4f}")
print(classification_report(
    vl_true, vl_pred, target_names=class_names, zero_division=0, digits=4
))
print(f"Macro F1:    {f1_score(vl_true, vl_pred, average='macro',    zero_division=0):.4f}")
print(f"Weighted F1: {f1_score(vl_true, vl_pred, average='weighted', zero_division=0):.4f}")

In [ ]:
te_loss, te_acc, _, _, _, te_spec, y_true, y_pred = evaluate(
    model, test_loader, criterion, DEVICE
)

print("\nTEST REPORT — HAM10000 Model 2 C6: Full FusionNet")
print(f"Loss:{te_loss:.4f} | Acc:{te_acc:.4f} | Spec:{te_spec:.4f}")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0, digits=4))
print(f"Macro F1:    {f1_score(y_true, y_pred, average='macro',    zero_division=0):.4f}")
print(f"Weighted F1: {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
print(f"\nMacro → "
      f"Prec:{precision_score(y_true, y_pred, average='macro', zero_division=0):.4f} | "
      f"Rec:{recall_score(y_true, y_pred, average='macro', zero_division=0):.4f} | "
      f"F1:{f1_score(y_true, y_pred, average='macro', zero_division=0):.4f} | "
      f"Spec:{te_spec:.4f}")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, square=True)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("HAM10000 Test CM — Model 2 C6: Full FusionNet")
plt.tight_layout(); plt.show()

all_train = train_losses + fine_train_losses + calib_train_losses
all_val   = val_losses   + fine_val_losses   + calib_val_losses
ep        = range(1, len(all_train) + 1)
p1_end    = len(train_losses)
p2_end    = p1_end + len(fine_train_losses)
best_ep   = all_val.index(min(all_val)) + 1

plt.figure(figsize=(10, 5))
plt.plot(ep, all_train, label="Train Loss")
plt.plot(ep, all_val,   label="Val Loss")
plt.axvline(p1_end,  color="gray",   linestyle="--", lw=1, label=f"Phase 1 End (Ep {p1_end})")
plt.axvline(p2_end,  color="orange", linestyle="--", lw=1, label=f"Phase 2 End (Ep {p2_end})")
plt.axvline(best_ep, color="green",  linestyle="-",  lw=1.2, label=f"Best Epoch ({best_ep})")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Loss Curves — HAM10000 Model 2 C6: Full FusionNet")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# Bootstrap CI
N_BOOTSTRAP = 1000
rng         = np.random.default_rng(SEED)
te_true_arr = np.array(y_true)
te_pred_arr = np.array(y_pred)
n_test      = len(te_true_arr)

boot = {"Accuracy": [], "F1 (Macro)": [], "Precision (Macro)": [], "Recall (Macro)": [], "Specificity": []}
per_class_f1 = {name: [] for name in class_names}

for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n_test, size=n_test)
    y_t, y_p = te_true_arr[idx], te_pred_arr[idx]
    boot["Accuracy"].append(accuracy_score(y_t, y_p))
    boot["F1 (Macro)"].append(f1_score(y_t, y_p, average="macro", zero_division=0))
    boot["Precision (Macro)"].append(precision_score(y_t, y_p, average="macro", zero_division=0))
    boot["Recall (Macro)"].append(recall_score(y_t, y_p, average="macro", zero_division=0))
    boot["Specificity"].append(compute_specificity(y_t, y_p, NUM_CLASSES))
    pc = f1_score(y_t, y_p, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    for ci, name in enumerate(class_names):
        per_class_f1[name].append(pc[ci])

def ci95(arr): return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

print("\n" + "=" * 72)
print(f"{'Metric':<25} {'Mean':>8}  {'Std':>8}  {'95% CI Lower':>14}  {'95% CI Upper':>14}")
print("=" * 72)
for metric, values in boot.items():
    arr = np.array(values); lo, hi = ci95(arr)
    print(f"{metric:<25} {arr.mean():>8.4f}  {arr.std():>8.4f}  {lo:>14.4f}  {hi:>14.4f}")
print("-" * 72)
print(f"\n{'Class':<52} {'Mean':>8}  {'Std':>8}  {'95% CI Lower':>14}  {'95% CI Upper':>14}")
print("-" * 72)
for name in class_names:
    arr = np.array(per_class_f1[name]); lo, hi = ci95(arr)
    print(f"{name:<52} {arr.mean():>8.4f}  {arr.std():>8.4f}  {lo:>14.4f}  {hi:>14.4f}")
print("=" * 72)

fig, axes = plt.subplots(1, 5, figsize=(18, 6))
fig.subplots_adjust(top=0.82, wspace=0.4)
for ax, (label, values) in zip(axes, boot.items()):
    arr = np.array(values); lo, hi = ci95(arr); mn, sd = arr.mean(), arr.std()
    ax.boxplot(arr, widths=0.5, patch_artist=True,
               boxprops=dict(facecolor="#4c72b0", alpha=0.75),
               medianprops=dict(color="white", linewidth=2),
               whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2),
               flierprops=dict(marker="o", markersize=3, markerfacecolor="gray", alpha=0.5))
    ax.set_title(label, fontsize=10, fontweight="bold", pad=8)
    ax.set_xticks([]); ax.set_ylabel("Score", fontsize=9)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.text(1, ax.get_ylim()[1], f"μ={mn:.4f}\nσ={sd:.4f}\n95%CI [{lo:.4f},{hi:.4f}]",
            ha="center", va="top", fontsize=7.5, color="#222222",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray", alpha=0.8))
plt.suptitle(f"Bootstrap Metric Distributions (n={N_BOOTSTRAP}) — Test Set",
             fontsize=13, fontweight="bold", y=0.97)
plt.savefig("/kaggle/working/bootstrap_metric_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

means  = np.array([np.mean(per_class_f1[n]) for n in class_names])
stds   = np.array([np.std(per_class_f1[n])  for n in class_names])
lo_arr = np.array([ci95(per_class_f1[n])[0] for n in class_names])
hi_arr = np.array([ci95(per_class_f1[n])[1] for n in class_names])

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(NUM_CLASSES)
bars = ax.bar(x, means, yerr=[means - lo_arr, hi_arr - means], capsize=6,
              color="#4c72b0", alpha=0.8, ecolor="#333333",
              error_kw={"linewidth": 1.8, "capthick": 1.8}, width=0.55)
ax.set_xticks(x)
ax.set_xticklabels([n.split("(")[-1].replace(")", "").strip() for n in class_names], fontsize=11)
ax.set_ylabel("F1 Score", fontsize=11); ax.set_ylim(0, 1.15)
ax.set_title("Per-Class F1 — Mean ± 95% CI (Bootstrap)", fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3, linestyle="--"); ax.spines[["top", "right"]].set_visible(False)
for bar, m, s, hi in zip(bars, means, stds, hi_arr):
    ax.text(bar.get_x() + bar.get_width() / 2, hi + 0.025,
            f"{m:.3f}\n±{s:.3f}", ha="center", va="bottom", fontsize=9, color="#111111")
plt.tight_layout()
plt.savefig("/kaggle/working/bootstrap_perclass_f1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def visualize_attention(model, loader, class_names, num_images=20):
    model.eval()
    collected_images, collected_preds, collected_labels, collected_attn = [], [], [], []
    collected = 0

    with torch.no_grad():
        for images, input_ids, attention_mask, tab_data, labels in loader:
            if collected >= num_images:
                break

            images         = images.to(DEVICE)
            input_ids      = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            tab_data       = tab_data.to(DEVICE)

            preds, attn_weights = model(images, input_ids, attention_mask, tab_data, return_attn=True)
            preds = torch.argmax(preds, dim=1).cpu()

            need = num_images - collected
            take = min(need, images.size(0))

            collected_images.append(images[:take].cpu())
            collected_preds.append(preds[:take])
            collected_labels.append(labels[:take])
            collected_attn.append(attn_weights[:take].cpu())

            collected += take

    images       = torch.cat(collected_images, dim=0)
    preds        = torch.cat(collected_preds, dim=0)
    labels       = torch.cat(collected_labels, dim=0)
    attn_weights = torch.cat(collected_attn, dim=0)

    n = len(images)  # will equal num_images unless the whole test set is smaller than that
    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    if n == 1:
        axes = axes.reshape(1, -1)   # keep 2D indexing consistent for a single row

    mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])

    for i in range(n):
        img = np.clip(std * images[i].permute(1, 2, 0).numpy() + mean, 0, 1)
        attn_map = cv2.resize(
            attn_weights[i].mean(dim=0).reshape(8, 8).numpy(),
            (img.shape[1], img.shape[0])
        )
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"True: {class_names[labels[i].item()]}", fontsize=10)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(img)
        axes[i, 1].imshow(attn_map, cmap='jet', alpha=0.5)
        axes[i, 1].set_title(f"Pred: {class_names[preds[i].item()]}", fontsize=10)
        axes[i, 1].axis('off')

    plt.tight_layout(); plt.show()

visualize_attention(model, test_loader, class_names, num_images=20)

In [ ]:
# =====================================================
# t-SNE VISUALIZATION OF FUSIONNET MULTIMODAL EMBEDDINGS (HAM10000)
# =====================================================

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

def extract_fusion_features(model, loader, device):
    model.eval()
    all_features, all_labels = [], []

    with torch.no_grad():
        for images, input_ids, attention_mask, meta_data, labels in loader:
            images         = images.to(device)
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            meta_data      = meta_data.to(device)

            # VISION
            f_convnext = model.convnext_features(images).flatten(2).transpose(1, 2)
            f_swin     = model.swin_norm(model.swin_features(images)).flatten(1, 2)
            fused_vision = model.vision_fuse(torch.cat([f_convnext, f_swin], dim=-1))

            # TEXT
            text_feats = model.text_encoder(
                input_ids=input_ids, attention_mask=attention_mask
            ).last_hidden_state

            # SHARED PROJECTION SPACE
            v_emb = F.normalize(model.vis_proj(fused_vision) + model.pos_embed, p=2, dim=-1)
            t_emb = F.normalize(model.text_proj(text_feats),                   p=2, dim=-1)

            # BIDIRECTIONAL CROSS-ATTENTION
            t_attended, _ = model.text_to_vis_attn(query=t_emb, key=v_emb, value=v_emb)
            v_attended, _ = model.vis_to_text_attn(query=v_emb, key=t_emb, value=t_emb)

            t_attended_pooled = t_attended.mean(dim=1)
            v_attended_pooled = v_attended.mean(dim=1)
            global_text       = t_emb.mean(dim=1)
            global_vision     = v_emb.mean(dim=1)

            # TABULAR
            f_meta      = model.tabular_mlp(meta_data)
            f_meta_proj = model.meta_proj(f_meta)

            # GATED FUSION -> FINAL 256-D EMBEDDING
            modalities   = [t_attended_pooled, v_attended_pooled, global_text, global_vision, f_meta_proj]
            stacked      = torch.stack(modalities, dim=1)
            gate_weights = model.modal_gate(
                torch.cat(modalities, dim=-1)
            ).softmax(dim=-1).unsqueeze(-1)
            fused        = (stacked * gate_weights).sum(dim=1)  # [B, 256]

            all_features.append(fused.cpu().numpy())
            all_labels.append(labels.numpy())

    all_features = np.concatenate(all_features, axis=0)
    all_labels   = np.concatenate(all_labels, axis=0)
    return all_features, all_labels

# RUN EXTRACTION ON THE TEST SET
all_features, all_labels = extract_fusion_features(model, test_loader, DEVICE)
print("Feature Shape:", all_features.shape)

# APPLY t-SNE
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate=200,
    random_state=42,
    init='pca'
)
tsne_features = tsne.fit_transform(all_features)

# PLOT
plt.figure(figsize=(12, 10))
scatter = plt.scatter(
    tsne_features[:, 0], tsne_features[:, 1],
    c=all_labels, cmap='tab10', s=60, alpha=0.8
)
plt.legend(
    handles=scatter.legend_elements()[0],
    labels=class_names,
    title="Classes", fontsize=11
)
plt.title("t-SNE Visualization of FusionNet Multimodal Embeddings over HAM10000 Test Set", fontsize=16)
plt.xlabel("t-SNE Dimension 1", fontsize=14)
plt.ylabel("t-SNE Dimension 2", fontsize=14)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/tsne_plot_ham.pdf", dpi=600, bbox_inches='tight')
plt.show()